# Phase 0 — Setup

**Goal of this notebook:** get everything ready so the rest of the project can run on
a free GPU. By the end you'll have (1) the code, (2) the libraries, and (3) the
dataset saved permanently in your Google Drive.

### The three tools we use, in one sentence each
- **GitHub** — a website that stores our *code*. We copy ("clone") it into Colab.
- **Google Colab** — a free website that runs Python on Google's computers, *including a free GPU* (the fast chip that trains neural networks).
- **Google Drive** — your personal cloud storage. Colab forgets everything when you close it, so we park the *dataset* in Drive so it survives.

Run each cell below with **Shift+Enter**, top to bottom.

## Step 1 — Turn on the GPU

In the Colab menu: **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.
Then run the next cell to confirm the GPU is visible.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "— none (set Runtime → T4 GPU)")

## Step 2 — Get the code from GitHub

Paste your repository's URL below (it looks like
`https://github.com/YOUR_USERNAME/pm25-visual-aq.git`). Running this clones the code
the first time, and `git pull` grabs the latest version every other time.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os
if not os.path.isdir("/content/pm25-visual-aq"):
    !git clone $REPO_URL /content/pm25-visual-aq
%cd /content/pm25-visual-aq
!git pull
import sys; sys.path.insert(0, "/content/pm25-visual-aq")
print("Working in:", os.getcwd())

## Step 3 — Install the libraries

`requirements.txt` lists everything the project needs. On Colab, PyTorch is already
installed with GPU support, so pip will skip it and keep the GPU build.

In [ ]:
!pip install -q -r requirements.txt
print("Libraries installed.")

## Step 4 — Connect Google Drive

A pop-up will ask you to allow access. This lets us save the dataset somewhere
permanent so we only download it once.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive connected.")

## Step 5 — Load the dataset and save it to Drive (run once)

This downloads PM25Vision (~1 GB) on Google's fast network and saves it to your
Drive. The next time you run it, it detects the saved copy and skips the download.

In [ ]:
from datasets import load_dataset, load_from_disk
from src.config import load_config

cfg = load_config()
drive_path = cfg["data"]["drive_path"]

if os.path.exists(drive_path):
    print("Already saved at", drive_path)
    ds = load_from_disk(drive_path)
else:
    print("Downloading", cfg["data"]["hf_repo"], "…")
    ds = load_dataset(cfg["data"]["hf_repo"])
    ds.save_to_disk(drive_path)
    print("Saved to", drive_path)

print(ds)

## Step 6 — Confirm it worked

We print the columns and a sample image with its AQI label. You should see ~11,219
rows total and a street photo.

In [ ]:
import numpy as np, io
from PIL import Image
import matplotlib.pyplot as plt

train = ds["train"]
pm = np.array(train["pm25"])
print("columns:", train.column_names)
print("rows: train=%d test=%d" % (len(ds["train"]), len(ds["test"])))
print("pm25 (AQI) min/median/max: %.0f / %.0f / %.0f" % (pm.min(), np.median(pm), pm.max()))

row = train[0]
img = row["image"]
img = img if isinstance(img, Image.Image) else Image.open(io.BytesIO(img)).convert("RGB")
plt.imshow(img); plt.title("pm25 (AQI) = %.0f" % row["pm25"]); plt.axis("off"); plt.show()

## Done ✓

You now have the code, the libraries, and the dataset in Drive. **Next:** open
`notebooks/01_data_audit.ipynb`. If you close Colab and come back later, just re-run
Steps 1–4 (the dataset is already saved, so Step 5 is instant).